<div style="border-top:4px solid #0f766e;padding:16px">DATA WAREHOUSING WITH APACHE DORIS · OPTIONAL LAB</div>

# 选做 Lab 5B：MySQL → Flink CDC → Doris

目标：验证快照、增删改、Checkpoint，以及停止后从 Savepoint 恢复并追平停机期间的变更。
约 35–50 分钟，Module 7 可复用本实验，不影响主线完成。

先阅读[环境说明](../../environments/streaming/README.md)。会启动课程 MySQL、Flink JobManager / TaskManager，复用 Doris 沙箱。
仅重建 `dw_course_l1_streaming.ext_cdc_orders`，清空课程 MySQL 的 `course_cdc.orders` 后写入三笔模拟订单。
不是 WWI 历史订单，不使用主线金额基线。同一沙箱不能多人并发执行；使用演示账号，不连接生产环境。
本例是**固定 Schema 单表 SQL 同步**，不宣称整库同步或自动 Schema 演进。

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
course = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dw_course").is_dir())
sys.path.insert(0, str(course))
from dw_course.docker_runtime import prepare_environment, connect_sandbox
from dw_course.runtime import expect
from dw_course.streaming import (
    prepare_streaming, kafka, produce, mysql, flink_api, submit_sql,
    wait_checkpoint, stop_with_savepoint, wait_rows,
)
prepare_environment(start=True)
os.environ["DW_DATABASE"] = "dw_course_l1_streaming"
lab = connect_sandbox()
print("独立实验库：", lab.database)

prepare_streaming("cdc", start=True)

## 1. 确认无运行任务，初始化源和目标

只在新实验开始时重置。恢复步骤绝不能再次执行这一节，否则会破坏源数据或目标状态。
源端配置 ROW/FULL Binlog，CDC 用户具有读取及复制权限；目标使用 MoW Unique Key。

In [ ]:
active = [j for j in flink_api("/jobs/overview")["jobs"] if j["state"] not in {"FINISHED","CANCELED","FAILED"}]
expect(active, [])
print(mysql("""DELETE FROM course_cdc.orders;
INSERT INTO course_cdc.orders VALUES
(920001,1,'CREATED',100.00),(920002,2,'CREATED',200.00),(920003,3,'CREATED',50.00);
SHOW MASTER STATUS;
SELECT * FROM course_cdc.orders ORDER BY order_id;"""))
lab.execute("DROP TABLE IF EXISTS ext_cdc_orders")
lab.execute("""CREATE TABLE ext_cdc_orders (
    order_id BIGINT NOT NULL, customer_id BIGINT NOT NULL,
    order_status VARCHAR(32) NOT NULL, order_amount DECIMAL(18,2) NOT NULL
) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")""")

## 2. 提交全量 + 增量 SQL

每次新实验使用唯一的 Sink label 前缀，恢复时保持同一份 SQL、并行度和前缀。
开启 Checkpoint 和 Sink 2PC；提交 INSERT 后任务持续运行，不能只凭 RUNNING 判断数据已同步。
初次构建下载三个 JAR，版本见环境 README。

In [ ]:
label = "course_cdc_" + uuid4().hex[:12]
sql = f"""SET 'execution.checkpointing.interval' = '5s';
SET 'parallelism.default' = '1';
SET 'pipeline.name' = 'course_mysql_orders';
CREATE TABLE mysql_orders (
 order_id BIGINT, customer_id BIGINT, order_status STRING, order_amount DECIMAL(18,2),
 PRIMARY KEY(order_id) NOT ENFORCED
) WITH (
 'connector'='mysql-cdc','hostname'='mysql','port'='3306',
 'username'='course_cdc','password'='course_cdc_local_only',
 'database-name'='course_cdc','table-name'='orders','server-id'='55101-55102',
 'server-time-zone'='Asia/Shanghai','scan.startup.mode'='initial'
);
CREATE TABLE doris_orders (
 order_id BIGINT, customer_id BIGINT, order_status STRING, order_amount DECIMAL(18,2),
 PRIMARY KEY(order_id) NOT ENFORCED
) WITH (
 'connector'='doris','fenodes'='doris:8030','benodes'='doris:8040','auto-redirect'='false','table.identifier'='dw_course_l1_streaming.ext_cdc_orders',
 'username'='root','password'='', 'sink.label-prefix'='{label}',
 'sink.enable-delete'='true','sink.enable-2pc'='true',
 'sink.properties.format'='json','sink.properties.read_json_by_line'='true'
);
INSERT INTO doris_orders SELECT order_id,customer_id,order_status,order_amount FROM mysql_orders;
"""
print(sql)
job_id = submit_sql(sql)
print("Job ID:", job_id)

In [ ]:
baseline = [[920001,1,"CREATED","100.00"],[920002,2,"CREATED","200.00"],[920003,3,"CREATED","50.00"]]
expect(wait_rows(lab,"ext_cdc_orders",baseline),baseline)
print(wait_checkpoint(job_id)["latest"]["completed"])
lab.sql("SELECT * FROM ext_cdc_orders ORDER BY order_id", title="快照结果")

## 3. 修改源端，验证真实 Binlog 增删改

更新 920001，删除 920002，新增 920004。删除来自源端事件，不是 Notebook 在 Doris 中执行 DELETE。
期望三行 / 225.00；同时查看 MySQL 当前数据与 Binlog 文件位置。

In [ ]:
print(mysql("""UPDATE course_cdc.orders SET order_status='PAID' WHERE order_id=920001;
DELETE FROM course_cdc.orders WHERE order_id=920002;
INSERT INTO course_cdc.orders VALUES (920004,4,'CREATED',75.00);
SELECT * FROM course_cdc.orders ORDER BY order_id;
SHOW MASTER STATUS;"""))
changed = [[920001,1,"PAID","100.00"],[920003,3,"CREATED","50.00"],[920004,4,"CREATED","75.00"]]
expect(wait_rows(lab,"ext_cdc_orders",changed),changed)
expect(lab.query("SELECT COUNT(*),SUM(order_amount) FROM ext_cdc_orders"),[[3,"225.00"]])

## 4. 保存状态并停止，源端继续写入

这是**受控 Savepoint 停止/恢复**，不是节点崩溃或自动故障转移实验。
Savepoint 包含源端消费进度与算子状态；`SHOW MASTER STATUS` 只是源端当前末尾位置，不能代替任务保存的消费位点。
不要清理 Binlog、MySQL 卷或 Flink 状态卷。

In [ ]:
savepoint = stop_with_savepoint(job_id)
print("恢复路径：", savepoint)
print(mysql("""UPDATE course_cdc.orders SET order_amount=125.00 WHERE order_id=920001;
DELETE FROM course_cdc.orders WHERE order_id=920003;
INSERT INTO course_cdc.orders VALUES (920005,5,'CREATED',40.00);
SHOW MASTER STATUS;"""))
expect(lab.query("SELECT * FROM ext_cdc_orders ORDER BY order_id"), changed)

## 5. 从保存状态恢复并追平

只增加恢复路径，复用原始 SQL；没有重新初始化源和目标。
期望三行 / 240.00，并保留 PAID 状态。恢复成功还要核对 Flink 的 restored 信息，而非只看最终行数。

In [ ]:
restored_job = submit_sql(f"SET 'execution.savepoint.path' = '{savepoint}';\n" + sql)
final = [[920001,1,"PAID","125.00"],[920004,4,"CREATED","75.00"],[920005,5,"CREATED","40.00"]]
expect(wait_rows(lab,"ext_cdc_orders",final),final)
checkpoints = wait_checkpoint(restored_job)
restored = checkpoints["latest"]["restored"]
expect(restored["external_path"],savepoint)
print("恢复记录：",restored)
expect(lab.query("SELECT COUNT(*),SUM(order_amount) FROM ext_cdc_orders"),[[3,"240.00"]])
print(mysql("SELECT * FROM course_cdc.orders ORDER BY order_id"))
lab.sql("SELECT * FROM ext_cdc_orders ORDER BY order_id", title="恢复后结果")

## 6. 停止本次任务，保留证据

提交：初始快照、增删改结果、Savepoint 路径、restored 信息和最终结果。
本实验覆盖正常运行与受控恢复，不证明所有故障路径下 exactly-once，也不覆盖乱序、多表事务原子可见或生产并发。

独立练习：停止后再修改一笔订单，从新 Savepoint 恢复，解释为何不能仅用订单的业务更新时间替代消费位点。
完成后按 README 停止课程服务；不自动删除数据卷。

In [ ]:
final_savepoint = stop_with_savepoint(restored_job)
print("最终保存路径：",final_savepoint)
lab.close()